# AutoMate AI 🚗 — Day 3: First Basic Agent

### Your Journey. Your Intelligent Co-Pilot.

This notebook follows the Day 3 class: build the first basic LangChain agent, understand the agent structure and ReAct loop, and test the agent against ten known questions. This stage has **no external tools connected yet**. Google Sheets and Telegram integrations are for a later session.

**Language:** AutoMate AI responds only in English.

> This is an educational prototype, not a live car database. It cannot fetch dealer prices, inventory, current specifications, recalls, or safety ratings.

## 1. Install the packages

In [ ]:
!pip -q install -U langchain langchain-openai langchain-google-genai pandas

## 2. Imports and provider configuration

Choose one provider. Add the corresponding API key to Colab Secrets and enable notebook access. The model identifiers are configurable and may change as providers update their catalogues.

In [ ]:
import os
import csv
import getpass
from pathlib import Path
from typing import Any

import pandas as pd
from google.colab import userdata, files
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage

# Choose: "google" or "openai"
MODEL_PROVIDER = "google"

# Provider model IDs; change these if your account/catalogue requires another model.
GEMINI_MODEL = "gemini-3.8-flash"
OPENAI_MODEL = "gpt-6.2-sol"

if MODEL_PROVIDER == "google":
    api_key = userdata.get("GOOGLE_API_KEY")
    if not api_key:
        raise ValueError("Add GOOGLE_API_KEY in Colab Secrets and enable notebook access.")
    os.environ["GOOGLE_API_KEY"] = api_key
    from langchain_google_genai import ChatGoogleGenerativeAI
    model = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0.3)

elif MODEL_PROVIDER == "openai":
    api_key = userdata.get("OPENAI_API_KEY")
    if not api_key:
        raise ValueError("Add OPENAI_API_KEY in Colab Secrets and enable notebook access.")
    os.environ["OPENAI_API_KEY"] = api_key
    from langchain_openai import ChatOpenAI
    model = ChatOpenAI(model=OPENAI_MODEL)

else:
    raise ValueError("MODEL_PROVIDER must be 'google' or 'openai'.")

print(f"Provider configured: {MODEL_PROVIDER}")

## 3. Understand the four parts of an agent

The class described an agent as four connected parts:

1. **Brain:** the language model.
2. **Instructions:** role, facts, rules, and response format.
3. **Tools:** functions or external services the agent may call. No external tools are connected in this Day 3 notebook.
4. **Memory:** conversation history retained for follow-up questions.

The agent works toward a goal rather than only returning a one-off answer. The ReAct idea is: reason, act, observe the result, and repeat when needed.

## 4. Define AutoMate AI instructions

In [ ]:
AUTOMATE_SYSTEM = """
You are AutoMate AI, an English-only AI sales and operations assistant for independent used-car dealerships.

GOAL
Help dealership staff understand customer enquiries, qualify buyer requirements, draft helpful responses, and think through lead-management workflows.

ROLE
You assist dealership owners, sales executives, and inventory coordinators. You do not replace their judgement or approval.

FACTS AND CURRENT LIMITATIONS
- This Day 3 learning agent has no external tools connected.
- You cannot access a dealership's live inventory, Google Sheets, CRM, messaging apps, or dealer systems.
- You cannot verify that a vehicle is available, reserve a vehicle, change stock, contact a buyer, or update a lead.
- You have no verified dealership-specific prices, vehicle details, policies, or customer records unless they are explicitly provided in the current conversation.

RULES
- Never invent stock availability, asking price, vehicle condition, mileage, ownership, accident history, service history, loan approval, or legal status.
- When asked about a specific vehicle in stock, explain that the live dealership inventory is not connected and ask staff to verify the approved inventory record.
- For customer enquiries, identify useful fields such as budget, preferred make/model, body type, fuel, transmission, location, purchase timeline, and contact preference.
- Ask concise follow-up questions when required details are missing.
- You may draft a customer reply or follow-up message, but clearly label it as a draft. Never claim that you sent it.
- You may explain how a lead or stock update should be recorded, but never claim that you changed an external record.
- Do not ask for passwords, OTPs, payment credentials, or unnecessary identity documents.
- Keep customer information private and recommend collecting only details needed for the enquiry.
- Be neutral and do not pressure buyers into a purchase.

FORMAT
- Respond only in English.
- Be professional, concise, and practical.
- Separate confirmed information from assumptions and items needing verification.
- Use clear headings or short bullet lists when useful.
"""

print("AutoMate AI instructions are ready.")

## 5. Create the first basic LangChain agent

This uses LangChain's `create_agent` function as shown in class. The agent is created with **an LLM and system instructions, but no external tools**. Tool-connected actions are intentionally left for the next lesson.

In [ ]:
agent = create_agent(
    model=model,
    tools=[],
    system_prompt=AUTOMATE_SYSTEM,
)

print("AutoMate AI basic agent created.")

## 6. Send a first request

In [ ]:
first_result = agent.invoke({
    "messages": [
        {"role": "user", "content": "I am a first-time car buyer. What information do you need before recommending a car?"}
    ]
})

first_answer = first_result["messages"][-1].content
print(first_answer)

## 7. Add simple conversation memory

This notebook keeps the latest eight question-and-answer turns in Python and sends that history with each new request. The history is in memory for this Colab session; it is not permanent storage.

In [ ]:
MAX_TURNS = 8
conversation_history = []

def ask_automate(question: str) -> str:
    """Ask AutoMate AI and retain up to eight recent question/answer turns."""
    if not isinstance(question, str) or not question.strip():
        raise ValueError("Please enter a non-empty question.")

    messages = conversation_history + [HumanMessage(content=question.strip())]
    result = agent.invoke({"messages": messages})
    answer = result["messages"][-1].content

    conversation_history.append(HumanMessage(content=question.strip()))
    conversation_history.append(AIMessage(content=answer))
    del conversation_history[:-MAX_TURNS * 2]
    return answer

def reset_memory() -> None:
    """Clear AutoMate AI's current-session conversation history."""
    conversation_history.clear()
    print("Conversation memory cleared.")

print(ask_automate("My name is Rohit and my budget is ₹10 lakh."))
print("\nStored messages:", len(conversation_history))
print(ask_automate("What is my name and budget?"))

## 8. Upload the ten-case test file

Download `Day-03/test.csv` from this GitHub repository to your computer, then upload it when prompted. The file contains the ten evaluation questions and expected keyword groups.

In [ ]:
uploaded = files.upload()

if not uploaded:
    raise ValueError("Please upload the Day-03 test.csv file.")

csv_name = next(iter(uploaded))
test_df = pd.read_csv(csv_name)

required_columns = {"question", "expected_groups"}
if not required_columns.issubset(test_df.columns):
    raise ValueError("CSV must contain question and expected_groups columns.")

print(f"Loaded {len(test_df)} test cases from {csv_name}.")
display(test_df)

## 9. Run all ten test cases

The quick evaluator checks whether at least one term from each expected group appears in the answer. This is only a preliminary keyword check. Read the answers yourself before accepting a PASS; a matching word alone does not prove an answer is correct.

In [ ]:
def evaluate_answer(answer: str, expected_groups: str) -> tuple[bool, str]:
    """Check that at least one alternative in each semicolon-separated group appears."""
    answer_lower = str(answer).lower()
    checks = []
    for group in str(expected_groups).split(";"):
        alternatives = [term.strip().lower() for term in group.split("|") if term.strip()]
        checks.append(any(term in answer_lower for term in alternatives))
    passed = all(checks) if checks else False
    return passed, f"{sum(checks)}/{len(checks)} groups matched"

test_results = []
for index, row in test_df.iterrows():
    question = str(row["question"])
    expected = str(row["expected_groups"])
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": question}]})
        answer = result["messages"][-1].content
        passed, note = evaluate_answer(answer, expected)
        status = "PASS" if passed else "REVIEW"
        error = ""
    except Exception as exc:
        answer = ""
        status = "ERROR"
        note = "Agent call failed"
        error = f"{type(exc).__name__}: {exc}"

    test_results.append({
        "case": index + 1,
        "question": question,
        "status": status,
        "check": note,
        "answer": answer,
        "error": error,
    })

results_df = pd.DataFrame(test_results)
display(results_df[["case", "status", "check", "question"]])
print("PASS:", int((results_df["status"] == "PASS").sum()))
print("REVIEW:", int((results_df["status"] == "REVIEW").sum()))
print("ERROR:", int((results_df["status"] == "ERROR").sum()))

## 10. Inspect the full answers and improve the instructions

Review every answer in the table below. Pay special attention to false claims about live prices, mileage, specifications, safety ratings, or external actions. If a case fails, improve the relevant facts or rules in `AUTOMATE_SYSTEM`, recreate the agent by rerunning Cell 5, and rerun the complete test set. Do not fix only one case without regression-testing the others.

In [ ]:
for _, row in results_df.iterrows():
    print("=" * 90)
    print(f"Case {row['case']} — {row['status']}")
    print("Question:", row["question"])
    print("Answer:", row["answer"])
    if row["error"]:
        print("Error:", row["error"])

## 11. Interactive AutoMate AI agent

Type `exit`, `quit`, or `bye` to stop. Type `reset` to clear conversation memory.

In [ ]:
print("🚗 AutoMate AI — Day 3 Basic Agent")
print("English-only automotive assistant. No external tools connected yet.")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit", "bye"}:
        print("AutoMate AI: Thanks for chatting. Drive safe!")
        break
    if user_input.lower() == "reset":
        reset_memory()
        continue
    try:
        print("\nAutoMate AI:", ask_automate(user_input), "\n")
    except Exception as exc:
        print(f"Agent error: {type(exc).__name__}: {exc}")
        print("Check your provider API key, model access, connection, and quota.")

## Day 3 submission checklist

- [ ] Choose Google Gemini or OpenAI and add its API key to Colab Secrets.
- [ ] Run the notebook from top to bottom.
- [ ] Upload `test.csv` and run all ten evaluation cases.
- [ ] Review all answers manually and note any failures.
- [ ] Improve the system instructions if needed and rerun all tests.
- [ ] Capture screenshots of the working agent and test results.
- [ ] Upload this notebook and `test.csv` to the repository's `Day-03` folder.

**Today's scope:** a basic LangChain agent without external tools. Google Sheets, Telegram, RAG, vector databases, and multi-agent orchestration are future lessons.